# Capítulo 2 — Análisis dimensional y similitud

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Qué significa exactamente que dos sistemas sean «semejantes»?

Integra el péndulo no lineal para muchas longitudes, gravedades y amplitudes.
Primero se dibuja el periodo crudo (un desastre de curvas) y después el
periodo adimensional frente a la amplitud (una sola curva).

La figura responde: ¿cuántos parámetros tiene realmente el problema?

Ejecutar:  python fig_colapso_pendulo.py

*(script original: `codigo/fig_colapso_pendulo.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp

from estilo_libro import C, save, use_style  # noqa: E402

use_style()


def periodo(longitud: float, gravedad: float, amplitud: float) -> float:
    """Periodo del péndulo simple no lineal, por cruce por cero."""
    def f(_t, y):
        return [y[1], -(gravedad / longitud) * np.sin(y[0])]

    def cruce(_t, y):
        return y[0]
    cruce.direction = 1.0          # sólo cruces ascendentes

    t_max = 20 * 2 * np.pi * np.sqrt(longitud / gravedad)
    sol = solve_ivp(f, (0, t_max), [amplitud, 0.0], events=cruce,
                    rtol=1e-10, atol=1e-12, dense_output=True)
    cruces = sol.t_events[0]
    return float(np.mean(np.diff(cruces))) if len(cruces) > 2 else np.nan


LONGITUDES = [0.25, 0.5, 1.0, 2.0]
GRAVEDADES = [1.62, 3.72, 9.81, 24.8]      # Luna, Marte, Tierra, Júpiter
amplitudes = np.linspace(0.05, 3.0, 24)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.2))
colores = [C.blue, C.red, C.green, C.ochre]

for L, color in zip(LONGITUDES, colores):
    for g, marca in zip(GRAVEDADES, ["o", "s", "^", "d"]):
        T = np.array([periodo(L, g, a) for a in amplitudes])
        ax1.plot(amplitudes, T, marca, color=color, ms=3, alpha=0.6)
        ax2.plot(amplitudes, T / (2 * np.pi * np.sqrt(L / g)), marca,
                 color=color, ms=3, alpha=0.6)

ax1.set_xlabel(r"amplitud inicial $\theta_0$ (rad)")
ax1.set_ylabel("periodo $T$ (s)")
ax1.set_title("16 combinaciones de $L$ y $g$: 16 curvas")
ax1.set_yscale("log")

# Curva teórica del colapso: T/T_0 = (2/pi) K(sin^2(theta0/2))
from scipy.special import ellipk  # noqa: E402
th = np.linspace(0.01, 3.0, 200)
ax2.plot(th, (2 / np.pi) * ellipk(np.sin(th / 2) ** 2), "-", color=C.ink,
         lw=1.8, label=r"$\frac{2}{\pi}K\!\left(\sin^2\frac{\theta_0}{2}\right)$")
ax2.axhline(1.0, color=C.grey, ls="--", lw=1.0)
ax2.text(0.1, 1.02, "aproximación de ángulo pequeño", color=C.grey, fontsize=8)
ax2.set_xlabel(r"amplitud inicial $\theta_0$ (rad)")
ax2.set_ylabel(r"$T\,/\,2\pi\sqrt{L/g}$")
ax2.set_title("Las mismas 16: una sola curva")
ax2.legend(loc="upper left")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué no hay mamíferos del tamaño de un edificio?

Compara el escalado geométrico (isometría) con el escalado que exige la
resistencia de los huesos, y contrasta con datos de secciones óseas reales.

La figura responde: si duplicas todas las longitudes de un animal, ¿resiste?

Ejecutar:  python fig_escala_huesos.py

*(script original: `codigo/fig_escala_huesos.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

masa = np.logspace(-2, 4, 200)          # de 10 g a 10 toneladas

# Isometría: toda longitud va como M^(1/3), toda área como M^(2/3)
area_iso = masa ** (2 / 3)
# Resistencia constante: el hueso debe soportar un peso proporcional a M,
# luego su sección debe ir como M^1
area_resistencia = masa ** 1.0

# Normalizamos ambas en M = 1 kg para poder compararlas
area_iso /= area_iso[np.argmin(abs(masa - 1))]
area_resistencia /= area_resistencia[np.argmin(abs(masa - 1))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.2))

ax1.loglog(masa, area_iso, color=C.blue, lw=2,
           label=r"isometría: $A \propto M^{2/3}$")
ax1.loglog(masa, area_resistencia, color=C.red, lw=2,
           label=r"tensión constante: $A \propto M^{1}$")
ax1.fill_between(masa, area_iso, area_resistencia,
                 where=area_resistencia > area_iso,
                 color=C.red, alpha=0.12)
ax1.annotate("déficit de hueso\nsi sólo escalas la forma",
             xy=(1e3, 1e2), xytext=(6, 3e3), fontsize=8.6, color=C.red,
             arrowprops=dict(arrowstyle="->", color=C.red, lw=1.0))
ax1.set_xlabel("masa corporal $M$ (kg)")
ax1.set_ylabel("sección del fémur (normalizada a 1 kg)")
ax1.set_title("Dos leyes de escala que divergen")
ax1.legend(loc="upper left")

# --- Tensión relativa que soportaría un animal isométrico ----------------
tension = masa / area_iso / (1 / area_iso[np.argmin(abs(masa - 1))])
tension = masa ** (1 - 2 / 3)
ax2.loglog(masa, tension, color=C.ochre, lw=2)
ax2.axhline(1, color=C.grey, ls="--", lw=1.0)
ax2.text(2e-2, 1.15, "tensión de un animal de 1 kg", color=C.grey, fontsize=8.2)
for m_ref, nombre in [(0.02, "ratón"), (70, "persona"),
                      (5e3, "elefante"), (1.5e5, "ballena\n(no camina)")]:
    if m_ref <= masa.max():
        ax2.plot(m_ref, m_ref ** (1 / 3), "o", color=C.ink, ms=5)
        ax2.annotate(nombre, (m_ref, m_ref ** (1 / 3)),
                     textcoords="offset points", xytext=(6, -2), fontsize=8)
ax2.set_xlabel("masa corporal $M$ (kg)")
ax2.set_ylabel(r"tensión relativa en el hueso $\propto M^{1/3}$")
ax2.set_title("Escalar la forma multiplica la tensión por $M^{1/3}$")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué una bacteria no puede nadar como un pez?

Mapa del número de Reynolds de nadadores y voladores, de la bacteria al
avión. La figura responde: ¿cuántas décadas de Re separan a los seres vivos, y
dónde está la frontera entre «manda la viscosidad» y «manda la inercia»?

Ejecutar:  python fig_mapa_reynolds.py

*(script original: `codigo/fig_mapa_reynolds.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# (nombre, longitud L en m, velocidad U en m/s, fluido)
CASOS = [
    ("Bacteria (E. coli)", 2e-6, 3e-5, "agua"),
    ("Espermatozoide", 5e-5, 2e-4, "agua"),
    ("Paramecio", 2e-4, 1e-3, "agua"),
    ("Larva de mosquito", 5e-3, 2e-2, "agua"),
    ("Renacuajo", 2e-2, 5e-2, "agua"),
    ("Sardina", 0.15, 1.0, "agua"),
    ("Persona nadando", 1.8, 1.5, "agua"),
    ("Atún", 2.0, 10.0, "agua"),
    ("Ballena azul", 25.0, 5.0, "agua"),
    ("Mosca de la fruta", 3e-3, 1.0, "aire"),
    ("Abeja", 1.3e-2, 5.0, "aire"),
    ("Gorrión", 0.15, 10.0, "aire"),
    ("Águila", 0.9, 20.0, "aire"),
    ("Airbus A320", 37.0, 250.0, "aire"),
]
NU = {"agua": 1.0e-6, "aire": 1.5e-5}      # viscosidad cinemática, m^2/s

fig, ax = plt.subplots(figsize=(8.6, 5.0))

for nombre, L, U, fluido in CASOS:
    Re = L * U / NU[fluido]
    color = C.blue if fluido == "agua" else C.ochre
    ax.plot(Re, L, "o", color=color, ms=7, zorder=4)
    ax.annotate(nombre, (Re, L), textcoords="offset points",
                xytext=(8, 4), fontsize=7.8, color=color)

# Fronteras de régimen
ax.axvspan(1e-8, 1, color=C.green, alpha=0.10)
ax.axvspan(1, 1e3, color=C.grey, alpha=0.10)
ax.axvspan(1e3, 1e10, color=C.red, alpha=0.07)
for x, txt, col in [(1e-4, "manda la viscosidad\n$Re \\ll 1$", C.green),
                    (30, "zona de nadie", C.grey),
                    (1e6, "manda la inercia\n$Re \\gg 1$", C.red)]:
    ax.text(x, 3e-6, txt, fontsize=8.6, color=col, ha="center", va="bottom")

ax.set_xscale("log"), ax.set_yscale("log")
ax.set_xlim(1e-6, 1e9), ax.set_ylim(1e-6, 2e2)
ax.set_xlabel("número de Reynolds  $Re = UL/\\nu$")
ax.set_ylabel("tamaño característico $L$ (m)")
ax.set_title("Quince décadas de Reynolds, dos mundos físicos distintos")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Se puede sacar la energía de una bomba de una fotografía y un cronómetro?

Ajusta la ley de onda de choque R = C (E t^2 / rho)^(1/5) a los radios de la
bola de fuego de Trinity publicados por G. I. Taylor (1950), obtenidos de las
fotografías de alta velocidad de J. E. Mack.

La figura responde: ¿es realmente una recta de pendiente 2/5 en log-log, y qué
energía sale de ella?

Ejecutar:  python fig_taylor_trinity.py

*(script original: `codigo/fig_taylor_trinity.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# Datos transcritos de la tabla publicada en Taylor (1950), parte II.
# t en milisegundos, R en metros. Son los radios medidos sobre las fotografías.
t_ms = np.array([0.10, 0.24, 0.38, 0.52, 0.66, 0.94, 1.25, 1.50, 1.93,
                 3.53, 4.61, 15.0, 25.0, 34.0, 53.0, 62.0])
R_m = np.array([11.1, 19.9, 25.4, 28.8, 31.9, 36.3, 41.0, 44.4, 46.9,
                59.0, 65.6, 106.5, 130.0, 145.0, 175.0, 185.0])
t = t_ms * 1e-3

RHO = 1.25      # densidad del aire ambiente, kg/m^3
C_TAYLOR = 1.03  # constante adimensional para gamma = 1.4 (Taylor, 1950)

# Ajuste de log R = a + m log t.  La teoría predice m = 2/5.
m, a = np.polyfit(np.log10(t), np.log10(R_m), 1)
# Energía a partir de cada punto, fijando la pendiente teórica
E_por_punto = RHO * R_m**5 / (C_TAYLOR**5 * t**2)
E = np.median(E_por_punto)

print(f"pendiente ajustada = {m:.3f}   (teoría: {2/5:.3f})")
print(f"E = {E:.2e} J = {E / 4.184e12:.1f} kt")
print(f"dispersión punto a punto: {E_por_punto.min()/1e12:.0f}–"
      f"{E_por_punto.max()/1e12:.0f} TJ")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.2))

# --- Panel 1: la ley de potencias ----------------------------------------
tt = np.logspace(-4.2, -1.1, 100)
ax1.loglog(t, R_m, "o", color=C.red, ms=6, label="Trinity (Taylor 1950)")
ax1.loglog(tt, 10**a * tt**m, "-", color=C.blue, lw=1.8,
           label=f"ajuste: $R \\propto t^{{{m:.2f}}}$")
# La teoría fija la pendiente en 2/5; sólo se ajusta la ordenada.
a_teoria = np.mean(np.log10(R_m) - 0.4 * np.log10(t))
ax1.loglog(tt, 10**a_teoria * tt**0.4, "--", color=C.green, lw=1.4,
           label="teoría: $R \\propto t^{2/5}$")
ax1.set_xlabel("tiempo desde la detonación (s)")
ax1.set_ylabel("radio del frente (m)")
ax1.set_title("Tres décadas de tiempo, una sola recta")
ax1.legend(loc="lower right")

# --- Panel 2: la energía deducida de cada punto ---------------------------
ax2.semilogx(t, E_por_punto / 4.184e12, "o", color=C.red, ms=6)
ax2.axhline(E / 4.184e12, color=C.blue, lw=1.8,
            label=f"mediana = {E / 4.184e12:.0f} kt")
ax2.axhspan(20, 22, color=C.green, alpha=0.18)
ax2.text(2e-4, 21, "valor aceptado hoy: ~21 kt", color=C.green, fontsize=8.6,
         va="center")
ax2.set_xlabel("tiempo desde la detonación (s)")
ax2.set_ylabel("energía deducida (kt de TNT)")
ax2.set_title("Cada foto da una energía. ¿Coinciden?")
ax2.set_ylim(0, 30)
ax2.legend(loc="lower right")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
